In [0]:
# -------------------------------------------------------------------------
# BRONZE LAYER QUALITY TESTS (DLT EDITION)
# Purpose: Verify DLT Ingestion, Audit Columns, and Error Handling
# Rubric: Domain 3 (Data Quality, Error Handling) & Domain 4 (Audit)
# -------------------------------------------------------------------------

from pyspark.sql.functions import col, count

# 1. Setup
catalog = "ecommerce_analytics_dev"
schema = "bronze_layer"
table_name = f"{catalog}.{schema}.events_raw"

print(f"🧪 Starting Quality Tests for: {table_name}")

# 2. TEST 1: Table Accessibility
try:
    df = spark.read.table(table_name)
    print("✅ TEST 1 PASSED: Table is accessible.")
except Exception as e:
    print(f"❌ TEST 1 FAILED: Cannot read table. Run the DLT pipeline first! Error: {e}")

# 3. TEST 2: Record Count (Sanity Check)
row_count = df.count()
if row_count > 0:
    print(f"✅ TEST 2 PASSED: Table is populated.")
else:
    print("❌ TEST 2 FAILED: Table is empty.")

# 4. TEST 3: Schema Validation (Critical Columns)
# We check for standard columns AND the DLT-specific _rescued_data
expected_columns = {
    "event_time": "timestamp",
    "user_id": "integer",
    "price": "double",
    "source_file": "string",          # Audit Column Check
    "ingestion_timestamp": "timestamp", # Audit Column Check
    "_rescued_data": "string"         # DLT Error Handling Check
}

print("--- Checking Schema ---")
current_schema = {f.name: f.dataType.typeName() for f in df.schema}
all_cols_pass = True

for col_name, expected_type in expected_columns.items():
    if col_name not in current_schema:
        print(f"❌ FAILED: Column '{col_name}' is MISSING.")
        all_cols_pass = False
    elif current_schema[col_name] != expected_type:
        # Note: _rescued_data might be complex string/struct, simplified check here
        print(f"⚠️ WARNING: Column '{col_name}' has type '{current_schema[col_name]}', expected '{expected_type}'.")
    else:
        print(f"   Column '{col_name}' exists and is {expected_type}.")

if all_cols_pass:
    print("✅ TEST 3 PASSED: Critical schema columns are present.")

# 5. TEST 4: Audit Column Verification
# Rubric Domain 4: "Raw data with audit columns"
null_audit_count = df.filter(col("ingestion_timestamp").isNull() | col("source_file").isNull()).count()

if null_audit_count == 0:
    print("✅ TEST 4 PASSED: Audit columns (ingestion_timestamp, source_file) are fully populated.")
else:
    print(f"❌ TEST 4 FAILED: Found {null_audit_count} rows with missing audit info.")

# 6. TEST 5: Volume Verification (13GB Check)
print(f"\n📊 Volume Check: {row_count:,.0f} total rows.")

if row_count > 1000000:
    print("✅ TEST 5 PASSED: Volume looks healthy (Millions of records).")
elif row_count == 0:
    print("❌ TEST 5 FAILED: Zero records found.")
else:
    print("⚠️ WARNING: Row count is low (< 1M). Did the stream process the full 13GB file?")

# 7. TEST 6: DLT Error Handling Check (_rescued_data)
# Rubric Domain 3: "Error Handling and Dead Letter Queues"
if "_rescued_data" in df.columns:
    rescued_count = df.filter(col("_rescued_data").isNotNull()).count()
    print(f"✅ TEST 6 PASSED: DLT Error handling column (_rescued_data) exists.")
    if rescued_count > 0:
        print(f"   ℹ️ Info: {rescued_count} rows were rescued (malformed data found and saved).")
    else:
        print(f"   ℹ️ Info: 0 rows rescued (Input data was perfectly formatted).")
else:
    print("❌ TEST 6 FAILED: _rescued_data column missing. DLT error handling not active.")

# 8. Sample Data Display
print("\n--- Sample Data (First 5 Rows) ---")
display(df.limit(5))

In [0]:
# -------------------------------------------------------------------------
# SILVER LAYER QUALITY TESTS
# Purpose: Verify Deduplication, Schema, and Quarantine Logic
# Rubric: Domain 3 (Data Quality) & Domain 4 (Silver Layer)
# -------------------------------------------------------------------------

from pyspark.sql.functions import col, count, sum as _sum

# 1. Setup
catalog = "ecommerce_analytics_dev"
schema = "silver_layer" # DLT writes here
silver_table = f"{catalog}.{schema}.events_cleaned" # Or events_silver depending on your table name
quarantine_table = f"{catalog}.{schema}.events_quarantine"

# NOTE: DLT tables are sometimes hidden or named differently depending on target schema settings.
# If "events_silver" is not found, check the "Target Schema" in your DLT settings.
# Based on your code, it should be 'events_silver'.

print(f"🧪 Starting Silver Tests...")

# 2. TEST 1: Silver Data Quality (The "Good" Data)
try:
    df_silver = spark.read.table("ecommerce_analytics_dev.bronze_layer.events_silver") 
    # NOTE: DLT often writes tables to the TARGET schema. 
    # If you set Target = bronze_layer in pipeline settings, they are there.
    # If you didn't set a target, they are in the 'Hive Metastore' or a random schema.
    # CHECK YOUR PIPELINE SETTINGS -> Storage -> Target Schema.
    # Assuming 'bronze_layer' based on your previous screenshots.
    
    print(f"✅ Accessing Silver Table...")
    
    # Check 1: Nulls (Should be ZERO because we dropped them)
    null_users = df_silver.filter(col("user_id").isNull()).count()
    if null_users == 0:
        print("✅ TEST PASSED: No null user_ids in Silver (Filter worked).")
    else:
        print(f"❌ TEST FAILED: Found {null_users} null user_ids!")

    # Check 2: Negative Prices
    neg_prices = df_silver.filter(col("price") < 0).count()
    if neg_prices == 0:
        print("✅ TEST PASSED: No negative prices in Silver.")
    else:
        print(f"❌ TEST FAILED: Found {neg_prices} negative prices!")

except Exception as e:
    print(f"⚠️ Could not read table. Check your Catalog/Schema name! Error: {e}")

# 3. TEST 2: Quarantine Validation
try:
    df_quarantine = spark.read.table("ecommerce_analytics_dev.bronze_layer.events_quarantine")
    q_count = df_quarantine.count()
    
    print(f"\n🗑️ Quarantine Check: {q_count} rows.")
    
    if q_count == 0:
        print("ℹ️ RESULT: Quarantine is empty. This means input data was 100% valid.")
        print("   (This is acceptable if the dataset is high quality).")
    else:
        print(f"✅ RESULT: Quarantine captured {q_count} bad rows.")
        display(df_quarantine.limit(5))
        
except Exception as e:
    print(f"⚠️ Could not read Quarantine table. Error: {e}")

# 4. TEST 3: Deduplication Verify
# If deduplication worked, we should not find any duplicate primary keys.
# This query groups by the Key and counts > 1. 
print("\n🔎 Checking for Duplicates in Silver (This takes time for 110M rows)...")
duplicate_count = (
    df_silver
    .groupBy("event_time", "user_id", "product_id", "event_type")
    .count()
    .filter(col("count") > 1)
    .count()
)

if duplicate_count == 0:
    print("✅ TEST PASSED: Zero duplicates found. Deduplication logic is perfect.")
else:
    print(f"❌ TEST FAILED: Found {duplicate_count} duplicate groups.")